In [1]:
%load_ext cudf.pandas

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("/kaggle/input/datasets/organizations/crowdflower/twitter-airline-sentiment/Tweets.csv")

In [3]:
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,None,nan,Virgin America,None,cairdin,None,0,@VirginAmerica What @dhepburn said.,None,2015-02-24 11:35:52 -0800,None,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,None,0.0,Virgin America,None,jnardino,None,0,@VirginAmerica plus you've added commercials t...,None,2015-02-24 11:15:59 -0800,None,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,None,nan,Virgin America,None,yvonnalynn,None,0,@VirginAmerica I didn't today... Must mean I n...,None,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,None,jnardino,None,0,@VirginAmerica it's really aggressive to blast...,None,2015-02-24 11:15:36 -0800,None,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0,Virgin America,None,jnardino,None,0,@VirginAmerica and it's a really big bad thing...,None,2015-02-24 11:14:45 -0800,None,Pacific Time (US & Canada)


## Phase 1 : EDA

### Checking for null values

In [4]:
print(f"TOTAL NUMS OF SAMPLES : {df.shape}\n\n")
print(f"TOTAL NUMS OF NULL VALUES : \n\n{df.isna().sum()}\n\n")
print(f"TOTAL NUMS OF NULL VALUES : \n\n{df.isna().mean()*100}\n")

TOTAL NUMS OF SAMPLES : (14640, 15)


TOTAL NUMS OF NULL VALUES : 

tweet_id                            0
airline_sentiment                   0
airline_sentiment_confidence        0
negativereason                   5462
negativereason_confidence        4118
airline                             0
airline_sentiment_gold          14600
name                                0
negativereason_gold             14608
retweet_count                       0
text                                0
tweet_coord                     13621
tweet_created                       0
tweet_location                   4733
user_timezone                    4820
dtype: int64


TOTAL NUMS OF NULL VALUES : 

tweet_id                         0.000000
airline_sentiment                0.000000
airline_sentiment_confidence     0.000000
negativereason                  37.308743
negativereason_confidence       28.128415
airline                          0.000000
airline_sentiment_gold          99.726776
name                   

Since feature "airline_sentiment_gold", "negativereason_gold", and "tweet_coord" are having more than 90 % of missing values, so we will be removing them.

In [5]:
df_1 = df.drop(columns=['tweet_coord','negativereason_gold','airline_sentiment_gold'])

In [6]:
# checking for remaining null values in df_1 
print(f"TOTAL FEATURES : {df_1.shape[1]}\n\n")
print(f"TOTAL NUMS OF NULL VALUES : \n\n{df_1.isna().sum()}\n\n")
print(f"TOTAL NUMS OF NULL VALUES : \n\n{df_1.isna().mean()*100}\n")

TOTAL FEATURES : 12


TOTAL NUMS OF NULL VALUES : 

tweet_id                           0
airline_sentiment                  0
airline_sentiment_confidence       0
negativereason                  5462
negativereason_confidence       4118
airline                            0
name                               0
retweet_count                      0
text                               0
tweet_created                      0
tweet_location                  4733
user_timezone                   4820
dtype: int64


TOTAL NUMS OF NULL VALUES : 

tweet_id                         0.000000
airline_sentiment                0.000000
airline_sentiment_confidence     0.000000
negativereason                  37.308743
negativereason_confidence       28.128415
airline                          0.000000
name                             0.000000
retweet_count                    0.000000
text                             0.000000
tweet_created                    0.000000
tweet_location                  32.3292

In [7]:
df_1.sample(3)

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,name,retweet_count,text,tweet_created,tweet_location,user_timezone
4998,569557305961750528,neutral,0.3491,None,0.0,Southwest,Million_Miler,0,"@SouthwestAir BTW, not a weather delay. We've ...",2015-02-22 10:00:18 -0800,None,None
4797,569727036627034112,neutral,1.0000,None,nan,Southwest,Barbee72,0,@SouthwestAir Just reading your boarding polic...,2015-02-22 21:14:45 -0800,None,None
7167,569923761232850944,neutral,1.0000,None,nan,Delta,LeroyFunkdafied,0,@JetBlue ...really?,2015-02-23 10:16:28 -0800,"Bangor, Maine, USA",Eastern Time (US & Canada)


In [8]:
df_1.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   tweet_id                      14640 non-null  int64
 1   airline_sentiment             14640 non-null  object
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object
 6   name                          14640 non-null  object
 7   retweet_count                 14640 non-null  int64
 8   text                          14640 non-null  object
 9   tweet_created                 14640 non-null  object
 10  tweet_location                9907 non-null   object
 11  user_timezone                 9820 non-null   object
dtypes: float64(2), int64(2), object(8)
memory usage: 3.5+ MB


In [9]:
df_1.columns.tolist()

['tweet_id',
 'airline_sentiment',
 'airline_sentiment_confidence',
 'negativereason',
 'negativereason_confidence',
 'airline',
 'name',
 'retweet_count',
 'text',
 'tweet_created',
 'tweet_location',
 'user_timezone']

### Changing the column names

In [10]:
dict = {
    'tweet_id':'id',
     'airline_sentiment':'sentiment',
     'airline_sentiment_confidence':'sentiment_confidence',
     'negativereason':'reason',
     'negativereason_confidence':'reason_confidence',
     'airline':'airlines',
     'name':'name',
     'retweet_count':'retweets',
     'text':'feedback',
     'tweet_created':'date_time',
     'tweet_location':'location',
     'user_timezone':'timezone'
    }


df_1 = df_1.rename(columns=dict)

In [11]:
df_1.columns.tolist()

['id',
 'sentiment',
 'sentiment_confidence',
 'reason',
 'reason_confidence',
 'airlines',
 'name',
 'retweets',
 'feedback',
 'date_time',
 'location',
 'timezone']

In [12]:
df_1 = df_1[['id','name','date_time', 'location', 'timezone','airlines','feedback','reason','reason_confidence','retweets','sentiment','sentiment_confidence']]
df_1.sample(5)

,id,name,date_time,location,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment,sentiment_confidence
1813,569571772355489792,aDeniseHuxtable,2015-02-22 10:57:47 -0800,Nueva York,Quito,United,@united called back and I've been on hold. Wha...,Customer Service Issue,1.0000,0,negative,1.0000
2051,569428333500375040,suntoshi,2015-02-22 01:27:48 -0800,"Bedford, Nh",Eastern Time (US & Canada),United,"@united is that all that matters, not the fact...",Lost Luggage,0.6837,0,negative,1.0000
4708,569911555967442944,Sara_Walsh,2015-02-23 09:27:58 -0800,None,Eastern Time (US & Canada),Southwest,@SouthwestAir yep. 99.99999999% certain it was...,Lost Luggage,1.0000,0,negative,1.0000
9807,569657128866095104,olfazo,2015-02-22 16:36:57 -0800,"New York, New York",None,US Airways,@USAirways where's our luggage? Been waiting m...,Lost Luggage,0.6583,0,negative,1.0000
2001,569481213016023040,jpfox13,2015-02-22 04:57:56 -0800,the district of colombia,Eastern Time (US & Canada),United,@united flight #1 no luck on #standby,Late Flight,0.3469,0,negative,0.6513


In [13]:
df_1.isna().sum()

id                         0
name                       0
date_time                  0
location                4733
timezone                4820
airlines                   0
feedback                   0
reason                  5462
reason_confidence       4118
retweets                   0
sentiment                  0
sentiment_confidence       0
dtype: int64

In [14]:
df_1.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id                    14640 non-null  int64
 1   name                  14640 non-null  object
 2   date_time             14640 non-null  object
 3   location              9907 non-null   object
 4   timezone              9820 non-null   object
 5   airlines              14640 non-null  object
 6   feedback              14640 non-null  object
 7   reason                9178 non-null   object
 8   reason_confidence     10522 non-null  float64
 9   retweets              14640 non-null  int64
 10  sentiment             14640 non-null  object
 11  sentiment_confidence  14640 non-null  float64
dtypes: float64(2), int64(2), object(8)
memory usage: 3.5+ MB


In [15]:
df_1.dtypes.unique()

array([dtype('int64'), dtype('O'), dtype('float64')], dtype=object)

In [16]:
df_1.dtypes[df_1.dtypes == 'object'].reset_index()

,index,0
0,name,object
1,date_time,object
2,location,object
3,timezone,object
4,airlines,object
5,feedback,object
6,reason,object
7,sentiment,object


In [17]:
df_1.dtypes[df_1.dtypes == 'int'].reset_index()

,index,0
0,id,int64
1,retweets,int64


In [18]:
df_1.dtypes[df_1.dtypes == 'float'].reset_index()

,index,0
0,reason_confidence,float64
1,sentiment_confidence,float64


In [19]:
pd.DataFrame({'Missing Values': df_1.isna().sum(), 'Data Type': df_1.dtypes})

,Missing Values,Data Type
id,0,int64
name,0,object
date_time,0,object
location,4733,object
timezone,4820,object
airlines,0,object
feedback,0,object
reason,5462,object
reason_confidence,4118,float64
retweets,0,int64


In [20]:
df_1.nunique()

id                      14485
name                     7701
date_time               14247
location                 3081
timezone                   85
airlines                    6
feedback                14427
reason                     10
reason_confidence        1410
retweets                   18
sentiment                   3
sentiment_confidence     1023
dtype: int64

In [21]:
import datetime as dt

In [22]:
# converting date_time into datetime format
df_1['date_time'] = pd.to_datetime(df_1['date_time'])
df_1.dtypes

id                                          int64
name                                       object
date_time               datetime64[ns, UTC-08:00]
location                                   object
timezone                                   object
airlines                                   object
feedback                                   object
reason                                     object
reason_confidence                         float64
retweets                                    int64
sentiment                                  object
sentiment_confidence                      float64
dtype: object

In [23]:
df_1['date'] = df_1['date_time'].dt.date
df_1['time'] = df_1['date_time'].dt.time

In [24]:
df_1 = df_1[['id','name','date','time', 'timezone','airlines','feedback','reason','reason_confidence','retweets','sentiment','sentiment_confidence']]
df_1.sample(5)

,id,name,date,time,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment,sentiment_confidence
7293,569685096527056896,mariambueno,2015-02-22,18:28:05,Quito,Delta,@JetBlue thank you!,None,NaN,0,positive,1.0
10690,568971304914853889,RhondaReger1,2015-02-20,19:11:44,None,US Airways,@USAirways would be less annoying if road cond...,Can't Tell,0.6292,0,negative,1.0
9013,570286345454800896,MauererPower,2015-02-24,10:17:14,Central Time (US & Canada),US Airways,@USAirways @AmericanAir how does a pilot forge...,Late Flight,0.6495,2,negative,1.0
13626,569790444797865984,KoolFatKat,2015-02-23,01:26:42,None,American,@AmericanAir Hi. I have KOA-LAX-PHL-ORD booked...,None,NaN,0,neutral,1.0
6334,568053732929359872,PaytonTaylor129,2015-02-18,06:25:38,Quito,Southwest,@SouthwestAir Landed in Nashville! Thanks for ...,None,NaN,1,positive,1.0


### Fixing the TimeZones

In [25]:
df_1['timezone'].unique().tolist()

['Eastern Time (US & Canada)',
 'Pacific Time (US & Canada)',
 'Central Time (US & Canada)',
 'America/New_York',
 'Atlantic Time (Canada)',
 'Quito',
 None,
 'Mountain Time (US & Canada)',
 'Vienna',
 'Caracas',
 'Kuala Lumpur',
 'Brisbane',
 'Arizona',
 'London',
 'Tehran',
 'Alaska',
 'Sydney',
 'Irkutsk',
 'Santiago',
 'Amsterdam',
 'Tijuana',
 'Abu Dhabi',
 'Central America',
 'Edinburgh',
 'Jerusalem',
 'Hawaii',
 'Paris',
 'Guam',
 'New Delhi',
 'Stockholm',
 'America/Chicago',
 'Berlin',
 'Madrid',
 'Athens',
 'Brussels',
 'Taipei',
 'Rome',
 'Beijing',
 'Mexico City',
 'Bern',
 'Singapore',
 'Indiana (East)',
 'Melbourne',
 'Saskatchewan',
 'Casablanca',
 'Brasilia',
 'Kyiv',
 'Bucharest',
 'Greenland',
 'Prague',
 'New Caledonia',
 'Bogota',
 'Seoul',
 'Sarajevo',
 'Wellington',
 'Bangkok',
 'Warsaw',
 'Copenhagen',
 'Hong Kong',
 'Guadalajara',
 'Mid-Atlantic',
 'Mazatlan',
 'Buenos Aires',
 'America/Los_Angeles',
 'Dublin',
 'Lisbon',
 'Newfoundland',
 'Monterrey',
 'Tokyo'

In [26]:
df_1['timezone'] = df_1['timezone'].str.replace(" (US & Canada)", "")
df_1['timezone'] = df_1['timezone'].str.replace(" ", "_")

In [27]:
# other way to do the same as above
timezone_fixes = {
    # Fixing the Eastern Time duplicates
    'America/New_York': 'Eastern_Time',
    'America/Detroit': 'Eastern_Time',
    'EST': 'Eastern_Time',
    'Indiana_(East)': 'Eastern_Time',
    
    # Fixing the Central Time duplicates
    'America/Chicago': 'Central_Time',
    
    # Fixing the Mountain Time duplicates
    'America/Boise': 'Mountain_Time',
    
    # Fixing the Pacific Time duplicates
    'America/Los_Angeles': 'Pacific_Time'
}

df_1['timezone'] = df_1['timezone'].replace(timezone_fixes)

In [28]:
df_1['timezone'].unique()

array(['Eastern_Time', 'Pacific_Time', 'Central_Time',
       'Atlantic_Time_(Canada)', 'Quito', None, 'Mountain_Time', 'Vienna',
       'Caracas', 'Kuala_Lumpur', 'Brisbane', 'Arizona', 'London',
       'Tehran', 'Alaska', 'Sydney', 'Irkutsk', 'Santiago', 'Amsterdam',
       'Tijuana', 'Abu_Dhabi', 'Central_America', 'Edinburgh',
       'Jerusalem', 'Hawaii', 'Paris', 'Guam', 'New_Delhi', 'Stockholm',
       'Berlin', 'Madrid', 'Athens', 'Brussels', 'Taipei', 'Rome',
       'Beijing', 'Mexico_City', 'Bern', 'Singapore', 'Melbourne',
       'Saskatchewan', 'Casablanca', 'Brasilia', 'Kyiv', 'Bucharest',
       'Greenland', 'Prague', 'New_Caledonia', 'Bogota', 'Seoul',
       'Sarajevo', 'Wellington', 'Bangkok', 'Warsaw', 'Copenhagen',
       'Hong_Kong', 'Guadalajara', 'Mid-Atlantic', 'Mazatlan',
       'Buenos_Aires', 'Dublin', 'Lisbon', 'Newfoundland', 'Monterrey',
       'Tokyo', 'Midway_Island', 'Istanbul', 'Solomon_Is.',
       'America/Atikokan', 'Adelaide', 'Nairobi', 'Lima', 'Is

### Fixing the Airlines

In [29]:
df_1['airlines'].unique()

array(['Virgin America', 'United', 'Southwest', 'Delta', 'US Airways',
       'American'], dtype=object)

In [30]:
df_1['airlines'] = df_1['airlines'].str.replace(" ", "_")

In [31]:
df_1['airlines'].unique()

array(['Virgin_America', 'United', 'Southwest', 'Delta', 'US_Airways',
       'American'], dtype=object)

In [32]:
pd.DataFrame({'MISSING VALUES':df_1.isna().sum(), 'DTYPES':df_1.dtypes})

,MISSING VALUES,DTYPES
id,0,int64
name,0,object
date,0,object
time,0,object
timezone,4820,object
airlines,0,object
feedback,0,object
reason,5462,object
reason_confidence,4118,float64
retweets,0,int64


In [33]:
df_1.sample(6)

,id,name,date,time,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment,sentiment_confidence
14117,569664668010225664,smb510,2015-02-22,17:06:55,Eastern_Time,American,@AmericanAir 719. Looks like we are about to g...,Late Flight,0.3849,0,negative,0.6701
9162,570085837821632512,worldwideweg,2015-02-23,21:00:30,None,US_Airways,"@USAirways Instead of fair treatment, I got a...",Flight Attendant Complaints,0.3425,0,negative,1.0000
10024,569566514132717568,SS8085,2015-02-22,10:36:53,Eastern_Time,US_Airways,"@USAirways After today,no reason 4 anyone to n...",Can't Tell,1.0000,0,negative,1.0000
6397,567883722948763648,librarymommy,2015-02-17,19:10:04,Eastern_Time,Southwest,@SouthwestAir Finally got through after 3 hour...,Late Flight,0.6702,0,negative,1.0000
7646,569460591644925952,suek03,2015-02-22,03:35:59,Eastern_Time,Delta,@JetBlue some woman stole my seat on the plane...,Bad Flight,0.6484,0,negative,1.0000
6106,568222976744824832,alana_aro,2015-02-18,17:38:09,None,Southwest,@SouthwestAir do you have all winners for @Ima...,None,NaN,0,neutral,0.6809


In [34]:
df_1['date'] = pd.to_datetime(df_1['date'])
df_1['hour'] = df_1['time'].apply(lambda x : x.hour)


In [35]:
df_1['hour'].unique()

array([11, 10,  9,  8,  7,  5, 23, 22, 21, 20, 18, 17, 16, 15, 14, 13, 12,
        6,  3,  0,  4,  2,  1, 19])

In [36]:
df_1 = df_1[['id','name','date','hour', 'timezone','airlines','feedback','reason','reason_confidence','retweets','sentiment']]


In [37]:
df_1.sample(6)

,id,name,date,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
938,570000364692484099,rodneyondrums,2015-02-23,15,Quito,United,@SouthwestAir doesn't charge ticket change fee...,Flight Booking Problems,0.6630,0,negative
3190,568607013212590080,Ozzy1652,2015-02-19,19,None,United,"@united I was rebooked, however it would have ...",Late Flight,0.6747,0,negative
1334,569815968194678784,8629Fissile,2015-02-23,3,London,United,@united Do you have a further update on the su...,Lost Luggage,0.6907,0,negative
7956,568930822952103938,KTrerotola,2015-02-20,16,Eastern_Time,Delta,@JetBlue just called that number and left mess...,Customer Service Issue,0.6376,0,negative
7743,569283532167757826,SpedAdvocates,2015-02-21,15,Eastern_Time,Delta,@JetBlue what crew? No one here is helping.,Customer Service Issue,0.6614,0,negative
14016,569677478077136897,CineDrones,2015-02-22,17,Eastern_Time,American,@AmericanAir it shouldn't happen but did and h...,Damaged Luggage,0.3734,0,negative


In [38]:
pd.DataFrame({'MISSING VALUES':df_1.isna().sum(), 'UNIQUE VALUES':df_1.nunique(), 'DTYPES':df_1.dtypes})

,MISSING VALUES,UNIQUE VALUES,DTYPES
id,0,14485,int64
name,0,7701,object
date,0,9,datetime64[ns]
hour,0,24,int64
timezone,4820,78,object
airlines,0,6,object
feedback,0,14427,object
reason,5462,10,object
reason_confidence,4118,1410,float64
retweets,0,18,int64


### Fixing Feedback feature

In [39]:
df_1.loc[:, ['airlines','feedback']].sample(20)

# as we can there is tag like "@....", so we have to remove every word starting with @

,airlines,feedback
5917,Southwest,@SouthwestAir I was just sitting here talking ...
12421,American,@AmericanAir Horrible service @loganairports. ...
3389,United,@united Case ID 8544484
8747,Delta,@JetBlue bag is supposedly here in Boston
14144,American,@AmericanAir I haven't been rebooked. I called...
8943,Delta,@JetBlue my request has nothing to do with res...
9056,US_Airways,@usairways Ok thank you…I call from Spain to U...
1869,United,@united can you assign seats
13242,American,@AmericanAir One hour to check in is 45 minute...
13697,American,"@AmericanAir seriously, there aren't any reps ..."


In [40]:
# repalcing tags
condition = r'@\w+ '
df_1['feedback'] = df_1['feedback'].str.replace(condition, "", regex=True)


In [41]:
df_1['feedback']

0                                               What said.
1        plus you've added commercials to the experienc...
2        I didn't today... Must mean I need to take ano...
3        it's really aggressive to blast obnoxious "ent...
4                 and it's a really big bad thing about it
                               ...                        
14635    thank you we got on a different flight to Chic...
14636    leaving over 20 minutes Late Flight. No warnin...
14637      Please bring American Airlines to #BlackBerry10
14638    you have my money, you change my flight, and d...
14639    we have 8 ppl so we need 2 know how many seats...
Name: feedback, Length: 14640, dtype: object

In [42]:
pd.DataFrame({'MISSING VALUES':df_1.isna().sum(), 'UNIQUE VALUES':df_1.nunique(), 'DTYPES':df_1.dtypes})

,MISSING VALUES,UNIQUE VALUES,DTYPES
id,0,14485,int64
name,0,7701,object
date,0,9,datetime64[ns]
hour,0,24,int64
timezone,4820,78,object
airlines,0,6,object
feedback,0,14340,object
reason,5462,10,object
reason_confidence,4118,1410,float64
retweets,0,18,int64


In [43]:
df_1.sample(6)

,id,name,date,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
11571,567925957090238464,fispahani,2015-02-17,21,Islamabad,US_Airways,Thanks. “@USAirways: Weather disruptions have ...,Late Flight,0.6677,1,negative
13273,569898392647811072,ephilp,2015-02-23,8,None,American,fl 249 to DFW is leaving Newark on time but is...,None,NaN,0,neutral
12116,570284308210032640,BartonDVM,2015-02-24,10,Central_Time,American,"Its not that I wasn't offered ""perks"" by @USAi...",Customer Service Issue,0.6632,0,negative
6898,570048890466111488,MariaZarkadas,2015-02-23,18,None,Delta,mco to lag two hour delay and sitting in Tarma...,Late Flight,1.0000,0,negative
13270,569899646509666304,gingermc23,2015-02-23,8,Eastern_Time,American,"confused at the definition of a ""preferred sea...",Bad Flight,0.6737,0,negative
14452,569614085807067136,macario2,2015-02-22,13,Brasilia,American,they did tell that our luggage stayed inside t...,Lost Luggage,0.3467,0,negative


In [44]:
df_1['reason'].unique()

array([None, 'Bad Flight', "Can't Tell", 'Late Flight',
       'Customer Service Issue', 'Flight Booking Problems',
       'Lost Luggage', 'Flight Attendant Complaints', 'Cancelled Flight',
       'Damaged Luggage', 'longlines'], dtype=object)

### Using KNN for handling missing values
- But before using KNN we need to apply OHE on timezone and reason

In [45]:
pd.DataFrame({'MISSING VALUES':df_1.isna().sum(), 'UNIQUE VALUES':df_1.nunique(), 'DTYPES':df_1.dtypes})

,MISSING VALUES,UNIQUE VALUES,DTYPES
id,0,14485,int64
name,0,7701,object
date,0,9,datetime64[ns]
hour,0,24,int64
timezone,4820,78,object
airlines,0,6,object
feedback,0,14340,object
reason,5462,10,object
reason_confidence,4118,1410,float64
retweets,0,18,int64


In [46]:
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

In [47]:
# printing only the nan column names
nan_cols = df_1.columns[df_1.isna().any()].tolist()
nan_cols

['timezone', 'reason', 'reason_confidence']

In [48]:
# grabbing all categorical cols
cat_cols = ['timezone', 'reason', 'airlines', 'sentiment']

# using OHE on all categorical cols
df_ohe_transformed = pd.get_dummies(df_1[cat_cols])
# here what happend is all NaN cells in timezone and reason have also been converted into columns due to OHE

In [49]:
condition = df_1[['timezone', 'reason']].isna()                                      # step - 1 : taking null values from timezone and reason

for col in ['timezone', 'reason']:
    temp_cols = [i for i in df_ohe_transformed.columns if i.startswith(col + '_')] 
    df_ohe_transformed.loc[condition[col], temp_cols] = np.nan                       # step - 2 : putting nan values at the empty cells

In [50]:
# combining rest of the numerical features with the ohe transformed features for better accuracy
numeric_cols = df_1[['reason_confidence', 'retweets', 'hour']]
all_features = pd.concat([df_ohe_transformed, numeric_cols], axis=1)

# applying KNN
knn = KNNImputer(n_neighbors=10, weights='distance')

imputed_array = knn.fit_transform(all_features)

df_imputed = pd.DataFrame(imputed_array, columns=all_features.columns, index=all_features.index)

In [52]:
imputed_array

array([[ 0.    ,  0.    ,  0.    , ...,  0.    ,  0.    , 11.    ],
       [ 0.    ,  0.    ,  0.    , ...,  0.    ,  0.    , 11.    ],
       [ 0.    ,  0.    ,  0.    , ...,  0.    ,  0.    , 11.    ],
       ...,
       [ 0.    ,  0.    ,  0.    , ...,  0.    ,  0.    , 11.    ],
       [ 0.    ,  0.    ,  0.    , ...,  0.6659,  0.    , 11.    ],
       [ 0.    ,  0.    ,  0.    , ...,  0.    ,  0.    , 11.    ]])

In [53]:
df_imputed.sample(6)

,timezone_Abu_Dhabi,timezone_Adelaide,timezone_Alaska,timezone_America/Atikokan,timezone_Amsterdam,timezone_Arizona,timezone_Athens,timezone_Atlantic_Time_(Canada),timezone_Bangkok,timezone_Beijing,...,airlines_Southwest,airlines_US_Airways,airlines_United,airlines_Virgin_America,sentiment_negative,sentiment_neutral,sentiment_positive,reason_confidence,retweets,hour
4846,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000000,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0000,0.0,18.0
6585,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000000,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0000,0.0,11.0
1497,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000000,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.6695,0.0,19.0
6003,0.0,0.0,0.0,0.0,0.0,1.0000,0.0,0.000000,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0000,0.0,7.0
13452,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0000,0.0,6.0
8721,0.0,0.0,0.0,0.0,0.0,0.0329,0.0,0.011093,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.6559,0.0,20.0


In [54]:
print(df_imputed.shape)

df_imputed.isna().sum().sum()

(14640, 100)


np.int64(0)

In [55]:
df_imputed.sample(10)

,timezone_Abu_Dhabi,timezone_Adelaide,timezone_Alaska,timezone_America/Atikokan,timezone_Amsterdam,timezone_Arizona,timezone_Athens,timezone_Atlantic_Time_(Canada),timezone_Bangkok,timezone_Beijing,...,airlines_Southwest,airlines_US_Airways,airlines_United,airlines_Virgin_America,sentiment_negative,sentiment_neutral,sentiment_positive,reason_confidence,retweets,hour
6482,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.34780,0.0,15.0
4482,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.68430,0.0,5.0
13085,0.0,0.0,0.0,0.0,0.000000,0.1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.00000,0.0,11.0
1598,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.00000,0.0,16.0
8629,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.40484,1.0,9.0
5551,0.0,0.0,0.0,0.0,0.106969,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.34410,4.0,14.0
10749,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.00000,0.0,15.0
2026,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.00000,0.0,3.0
12910,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.00000,0.0,14.0
14056,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.00000,0.0,17.0


In [56]:
reason_col = [c for c in df_imputed.columns if c.startswith('reason_') and c!= 'reason_confidence']
timezone_col = [c for c in df_imputed.columns if c.startswith('timezone_')]

In [57]:
temp = df_imputed[['reason_confidence']].copy()

# to convert OHE encoded values back into the normal feature we use 'idxmax(axis=1)' 
temp['reason'] = df_imputed[reason_col].idxmax(axis=1).str.replace('reason_','')
temp['timezone'] = df_imputed[timezone_col].idxmax(axis=1).str.replace('timezone_','')

temp

,reason_confidence,reason,timezone
0,0.0000,Flight Booking Problems,Eastern_Time
1,0.0000,Customer Service Issue,Pacific_Time
2,0.0000,Flight Attendant Complaints,Central_Time
3,0.7033,Bad Flight,Pacific_Time
4,1.0000,Can't Tell,Pacific_Time
...,...,...,...
14635,0.0000,Flight Booking Problems,Eastern_Time
14636,1.0000,Customer Service Issue,Central_Time
14637,0.0000,Flight Booking Problems,Mountain_Time
14638,0.6659,Customer Service Issue,Eastern_Time


In [58]:
temp.isna().sum()

reason_confidence    0
reason               0
timezone             0
dtype: int64

In [59]:
temp['reason_confidence'].describe()

count    14640.000000
mean         0.473663
std          0.391897
min          0.000000
25%          0.000000
50%          0.630200
75%          0.705000
max          1.000000
Name: reason_confidence, dtype: float64

In [60]:
# replacing the columns with nan values in df_1 with imputed values from temp
df_1[['reason_confidence', 'reason', 'timezone']] = temp[['reason_confidence', 'reason', 'timezone']]

In [61]:
pd.DataFrame({
    'NULL VAUES':df_1.isna().sum(),
    'UNIQUE VALUES':df_1.nunique(),
    'DTYPES':df_1.dtypes
})

,NULL VAUES,UNIQUE VALUES,DTYPES
id,0,14485,int64
name,0,7701,object
date,0,9,datetime64[ns]
hour,0,24,int64
timezone,0,78,object
airlines,0,6,object
feedback,0,14340,object
reason,0,10,object
reason_confidence,0,1831,float64
retweets,0,18,int64


In [62]:
df_1.select_dtypes(include='object')

,name,timezone,airlines,feedback,reason,sentiment
0,cairdin,Eastern_Time,Virgin_America,What said.,Flight Booking Problems,neutral
1,jnardino,Pacific_Time,Virgin_America,plus you've added commercials to the experienc...,Customer Service Issue,positive
2,yvonnalynn,Central_Time,Virgin_America,I didn't today... Must mean I need to take ano...,Flight Attendant Complaints,neutral
3,jnardino,Pacific_Time,Virgin_America,"it's really aggressive to blast obnoxious ""ent...",Bad Flight,negative
4,jnardino,Pacific_Time,Virgin_America,and it's a really big bad thing about it,Can't Tell,negative
...,...,...,...,...,...,...
14635,KristenReenders,Eastern_Time,American,thank you we got on a different flight to Chic...,Flight Booking Problems,positive
14636,itsropes,Central_Time,American,leaving over 20 minutes Late Flight. No warnin...,Customer Service Issue,negative
14637,sanyabun,Mountain_Time,American,Please bring American Airlines to #BlackBerry10,Flight Booking Problems,neutral
14638,SraJackson,Eastern_Time,American,"you have my money, you change my flight, and d...",Customer Service Issue,negative


In [64]:
print(sorted(df_1['hour'].unique().tolist()))
df_1['hour'].nunique()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


24

In [66]:
df_1['day_name'] = df_1['date'].dt.day_name()
df_1 = df_1[['id', 'name', 'date', 'day_name', 'hour', 'timezone', 'airlines', 'feedback', 'reason', 'reason_confidence', 'retweets', 'sentiment']]


In [67]:
df_1.sample(10)

,id,name,date,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
3683,568188203397574656,jjirsa,2015-02-18,Wednesday,15,Pacific_Time,United,"Honestly, I stopped trying to report things vi...",Late Flight,0.0000,0,positive
12596,570093652120178689,marcy_test,2015-02-23,Monday,21,Quito,American,no not yet. Waiting now to be connected to an ...,Customer Service Issue,0.6553,0,negative
13156,569925990283145216,iSmellNothing,2015-02-23,Monday,10,Eastern_Time,American,"I'd like to explore both options, and what the...",Customer Service Issue,0.0000,0,neutral
10158,569516370506993664,AndrewAquilante,2015-02-22,Sunday,7,Eastern_Time,US_Airways,why did you Cancelled Flight flight 1773 to ph...,Cancelled Flight,1.0000,0,negative
7374,569634575653183488,rambinasaurus,2015-02-22,Sunday,15,Atlantic_Time_(Canada),Delta,I think I have it selected already ☺️,Late Flight,0.0000,0,neutral
8516,568198057155641344,GavinRamblesOn,2015-02-18,Wednesday,15,Mountain_Time,Delta,I'd fly to an airline who actually gave a crap...,Flight Booking Problems,0.3868,0,negative
7982,568906811308257280,pseudomachine,2015-02-20,Friday,14,Eastern_Time,Delta,Thank you.,Late Flight,0.0000,0,neutral
2606,569005718365298689,ldellabella,2015-02-20,Friday,21,Eastern_Time,United,Of course. That was the start of my trip 3 wks...,Lost Luggage,0.6652,0,negative
344,568536882012749824,conorjrogers,2015-02-19,Thursday,14,Eastern_Time,Virgin_America,trying to check-in...but looks like your site ...,Customer Service Issue,0.3461,0,negative
10256,569443675194966016,drewdenker,2015-02-22,Sunday,2,London,US_Airways,now on hold for 90 minutes,Customer Service Issue,1.0000,0,negative


In [71]:
# SINCE WE ARE USING KAGGLE NOTEBOOK TO WRITE ALL THE CODES, SO HERE SAVING FILES WITH pickle WOULD BE DIFFICULT TO FIND AND TRICKY TO USE
# TO AVOID SUCH SCENARIO WE WILL BE USING FileLink and .kl FILE EXTENSION INSTEAD OF .pkl FILE EXTENSION

from IPython.display import FileLink
# This creates a clickable blue link in your Kaggle notebook!
FileLink('df_1.kl')

/kaggle/working/df_1.kl